<a href="https://colab.research.google.com/github/taijarals/trade_bot/blob/main/script_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
import json
import requests
import pandas as pd
from foxbit_api import API_BASE # Importa a base da API
import time
import hmac
import hashlib
import json
import requests
from urllib.parse import urlencode
import numpy as np
from openai import OpenAI


from google.colab import userdata
OPENROUTER_KEY = userdata.get('OPENROUTER_KEY')

# Carrega chave de ambiente
load_dotenv()

# Configura cliente para OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_KEY
)

In [44]:
from google.colab import userdata
FOXBIT_KEY = userdata.get('FOXBIT_KEY')
FOXBIT_SECRET = userdata.get('FOXBIT_SECRET')

API_BASE = "https://api.foxbit.com.br"



def gerar_assinatura(api_secret, method, path, params=None, body=""):
    """Gera a assinatura HMAC e cabeçalhos exigidos pela Foxbit."""

    timestamp = str(int(time.time() * 1000))

    queryString = urlencode(params) if params else ''

    rawBody = json.dumps(body) if body else ''

    preHash = f"{timestamp}{method.upper()}{path}{queryString}{rawBody}"

    assinatura = hmac.new(
        api_secret.encode(), preHash.encode(), hashlib.sha256
    ).hexdigest()

    headers = {
        "X-FB-ACCESS-KEY": FOXBIT_KEY,
        "X-FB-ACCESS-TIMESTAMP": timestamp,
        "X-FB-ACCESS-SIGNATURE": assinatura,
        "Content-Type": "application/json",
    }
    return headers


def chamada_api_privada(method, endpoint, payload=None, params=None):
    url = f"{API_BASE}{endpoint}"

    headers = gerar_assinatura(API_SECRET, method, endpoint, params, payload)

    try:
        response = requests.request(method, url, headers=headers, json=payload, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Erro ao acessar API: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print("📩 Resposta da API:", e.response.text)
        return None


In [45]:
def obter_ticker(symbol):
    """Obtém o preço atual de mercado (último preço) para o par informado."""
    endpoint = f"/rest/v3/markets/{symbol}/ticker/24hr"
    try:
        r = requests.get(API_BASE + endpoint)
        r.raise_for_status()
        data = r.json().get("data", {})

        df_ticker = pd.json_normalize(data)

        return df_ticker

    except Exception as e:
        print("❌ Erro ao consultar ticker:", e)
        return None


def pegar_candlesticks(symbol, interval, limit):
    endpoint = f"/rest/v3/markets/{symbol}/candlesticks"
    params = {"interval": interval, "limit": limit}

    response = requests.get(API_BASE + endpoint, params=params)
    if response.status_code != 200:
        print(f"❌ Erro ao obter candlesticks. Status code: {response.status_code}")
        print("Resposta bruta:", response.text)
        raise Exception("Erro na API ao obter candlesticks.")

    candles = response.json() # lista de listas

    df = pd.DataFrame(candles, columns=[
        "timestamp_open", "open", "high", "low", "close",
        "timestamp_close", "volume", "quoteVolume", "count",
        "takerBuyVolume", "takerBuyQuoteVolume"
    ])

    # Converter timestamps para datetime legível
    df["timestamp_open"] = pd.to_datetime(df["timestamp_open"], unit='ms')
    df["timestamp_close"] = pd.to_datetime(df["timestamp_close"], unit='ms')

    return df


def obter_orderbook(symbol):
    """Obtém o livro de ordens (orderbook) para o par informado."""
    endpoint = f"/rest/v3/markets/{symbol}/orderbook"

    params = {"level": 5}

    try:
        r = requests.get(API_BASE + endpoint)
        r.raise_for_status()
        data = r.json()

        # Transformando asks em DataFrame
        df_asks = pd.DataFrame(data['asks'], columns=['price', 'volume'])
        df_asks['type'] = 'ask'

        # Transformando bids em DataFrame
        df_bids = pd.DataFrame(data['bids'], columns=['price', 'volume'])
        df_bids['type'] = 'bid'

        df_order_book = pd.concat([df_asks, df_bids], ignore_index=True)

        return df_order_book

    except Exception as e:
        print("❌ Erro ao consultar orderbook:", e)
        return None


In [46]:
df_candles = pegar_candlesticks("btcbrl", "1m", "10")
df_ticker = obter_ticker("btcbrl")
df_orderbook = obter_orderbook("btcbrl")

/tmp/ipykernel_8576/2388361249.py:37: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df["timestamp_open"] = pd.to_datetime(df["timestamp_open"], unit='ms')
/tmp/ipykernel_8576/2388361249.py:38: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df["timestamp_close"] = pd.to_datetime(df["timestamp_close"], unit='ms')


In [47]:
def resumir_candles(df, n=20):
    import pandas as pd
    import numpy as np

    df = df.tail(n).copy()

    # garante tipo numérico
    df["close"] = pd.to_numeric(df["close"], errors="coerce")

    # remove possíveis NaNs
    df = df.dropna(subset=["close"])

    # validação mínima
    if len(df) < 2:
        return {
            "preco_atual": None,
            "retorno_curto_pct": None,
            "volatilidade_pct": None,
            "tendencia_curta": None,
            "rsi_14": None,
            "distancia_sma_20_pct": None
        }

    # =========================
    # PREÇOS
    # =========================
    preco_atual = df.iloc[-1]["close"]
    preco_inicial = df.iloc[0]["close"]

    retorno_pct = ((preco_atual - preco_inicial) / preco_inicial) * 100

    volatilidade_pct = df["close"].pct_change().std() * 100

    # =========================
    # RSI 14
    # =========================
    delta = df["close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window=14).mean()
    avg_loss = loss.rolling(window=14).mean()

    # evita divisão por zero sem perder pandas.Series
    rs = avg_gain / avg_loss.replace(0, np.nan)

    rsi = 100 - (100 / (1 + rs))

    # quando não houver perdas, RSI tende a 100
    rsi = rsi.fillna(100)

    rsi_atual = rsi.iloc[-1]

    # =========================
    # SMA 20
    # =========================
    sma_20 = (
        df["close"].rolling(window=20).mean().iloc[-1]
        if len(df) >= 20
        else np.nan
    )

    distancia_sma_pct = None

    if not pd.isna(sma_20):
        distancia_sma_pct = (
            ((preco_atual - sma_20) / sma_20) * 100
        )

    # =========================
    # TENDÊNCIA
    # =========================
    if retorno_pct > 0.2:
        tendencia = "alta"

    elif retorno_pct < -0.2:
        tendencia = "baixa"

    else:
        tendencia = "lateral"

    # =========================
    # RETORNO
    # =========================
    return {
        "preco_atual": round(float(preco_atual), 4),

        "retorno_curto_pct": round(float(retorno_pct), 3),

        "volatilidade_pct": (
            None
            if pd.isna(volatilidade_pct)
            else round(float(volatilidade_pct), 3)
        ),

        "tendencia_curta": tendencia,

        "rsi_14": (
            None
            if pd.isna(rsi_atual)
            else round(float(rsi_atual), 2)
        ),

        "distancia_sma_20_pct": (
            None
            if distancia_sma_pct is None or pd.isna(distancia_sma_pct)
            else round(float(distancia_sma_pct), 3)
        )
    }

In [48]:


def resumir_ticker_df(df_ticker):
    last = df_ticker.iloc[-1]

    last_price = float(last["last_trade.price"])
    bid_price = float(last["best.bid.price"])
    ask_price = float(last["best.ask.price"])

    spread_pct = ((ask_price - bid_price) / last_price) * 100

    return {
        "preco_atual": last_price,
        "spread_pct": round(spread_pct, 4),
        "variacao_24h_pct": float(last["rolling_24h.price_change_percent"]),
        "volume_24h": float(last["rolling_24h.volume"]),
        "trades_24h": int(last["rolling_24h.trades_count"]),
        "max_24h": float(last["rolling_24h.high"]),
        "min_24h": float(last["rolling_24h.low"])
    }

def pressao_bid_ask_df(df_ticker):
    last = df_ticker.iloc[-1]

    bid_vol = float(last["best.bid.volume"])
    ask_vol = float(last["best.ask.volume"])

    if ask_vol == 0:
        razao = None
    else:
        razao = bid_vol / ask_vol

    return {
        "bid_ask_ratio_top": None if razao is None else round(razao, 3)
    }

def resumir_orderbook_df(df_orderbook, top_n=5):
    asks = df_orderbook[df_orderbook["type"] == "ask"].head(top_n)
    bids = df_orderbook[df_orderbook["type"] == "bid"].head(top_n)

    vol_asks = asks["volume"].astype(float).sum()
    vol_bids = bids["volume"].astype(float).sum()

    if vol_asks == 0:
        pressao = None
    else:
        pressao = vol_bids / vol_asks

    return {
        "volume_asks_top": round(float(vol_asks), 6),
        "volume_bids_top": round(float(vol_bids), 6),
        "pressao_compra": None if pressao is None else round(float(pressao), 3)
    }

def resumir_orderbook_ponderado(df_orderbook, preco_atual, top_n=10):
    df = df_orderbook.copy()

    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["volume"] = pd.to_numeric(df["volume"], errors="coerce")

    asks = df[df["type"] == "ask"].head(top_n)
    bids = df[df["type"] == "bid"].head(top_n)

    # Evitar divisão por zero ou valores não numéricos em 'price'
    asks["peso"] = preco_atual / asks["price"].replace(0, np.nan) # replace 0 with NaN to avoid division by zero
    bids["peso"] = bids["price"] / preco_atual

    volume_asks_ponderado = (asks["volume"] * asks["peso"]).sum()
    volume_bids_ponderado = (bids["volume"] * bids["peso"]).sum()

    total = volume_asks_ponderado + volume_bids_ponderado

    return {
        "pressao_compra_ponderada": (
            volume_bids_ponderado / total if total > 0 else None
        )
    }

def montar_contexto_mercado(df_candles, df_ticker, df_orderbook):
    contexto = {}

    contexto.update(resumir_candles(df_candles))
    contexto.update(resumir_ticker_df(df_ticker))
    contexto.update(pressao_bid_ask_df(df_ticker))
    contexto.update(resumir_orderbook_df(df_orderbook))
    # Check if 'preco_atual' exists before calling resumir_orderbook_ponderado
    if "preco_atual" in contexto:
        contexto.update(resumir_orderbook_ponderado(df_orderbook, contexto["preco_atual"])) # Pass preco_atual
    else:
        print("""⚠️ 'preco_atual' not found in context. Cannot calculate 'pressao_compra_ponderada'.""")

    contexto["ativo"] = "BTC/BRL"
    contexto["timeframe"] = "1m"

    return contexto


In [49]:
contexto_mercado = montar_contexto_mercado(
    df_candles,
    df_ticker,
    df_orderbook
)

contexto_mercado

{'preco_atual': 388492.0,
 'retorno_curto_pct': -0.022,
 'volatilidade_pct': 0.031,
 'tendencia_curta': 'lateral',
 'rsi_14': 100.0,
 'distancia_sma_20_pct': None,
 'spread_pct': 0.0252,
 'variacao_24h_pct': -0.15677123,
 'volume_24h': 9.86550704,
 'trades_24h': 8732,
 'max_24h': 389878.0,
 'min_24h': 383693.0,
 'bid_ask_ratio_top': 31.84,
 'volume_asks_top': 0.06011,
 'volume_bids_top': 0.783192,
 'pressao_compra': 13.029,
 'pressao_compra_ponderada': np.float64(0.8328441225300945),
 'ativo': 'BTC/BRL',
 'timeframe': '1m'}

In [53]:

# ==============================================================
# 🧠 INTELIGÊNCIA ARTIFICIAL
# ==============================================================

def montar_prompt_ia(contexto):
    prompt = f"""
Você é um analista quantitativo especializado em mercado financeiro,
microestrutura e fluxo de ordens.

Avalie o estado atual do mercado e decida a melhor ação entre:
COMPRAR, VENDER ou ESPERAR.

Contexto atual do mercado:

Ativo: {contexto['ativo']}
Timeframe: {contexto['timeframe']}

Preço atual: {contexto['preco_atual']}
Retorno curto (%): {contexto['retorno_curto_pct']}
Volatilidade (%): {contexto['volatilidade_pct']}
Tendência curta: {contexto['tendencia_curta']}

Variação 24h (%): {contexto['variacao_24h_pct']}
Máxima 24h: {contexto['max_24h']}
Mínima 24h: {contexto['min_24h']}
Volume 24h: {contexto['volume_24h']}
Trades 24h: {contexto['trades_24h']}

Spread (%): {contexto['spread_pct']}

Order Book:
- Volume bids (top): {contexto['volume_bids_top']}
- Volume asks (top): {contexto['volume_asks_top']}
- Bid/Ask ratio: {contexto['bid_ask_ratio_top']}
- Pressão de compra: {contexto['pressao_compra']}
- Pressão de compra ponderada: {contexto['pressao_compra_ponderada']}

Responda exclusivamente no formato JSON abaixo:

{{
  "acao": "COMPRAR | VENDER | ESPERAR",
  "confianca": 0.0,
  "justificativa_curta": ""
}}
"""
    return prompt

def analisar_mercado(contexto_mercado, client):
    prompt_final = montar_prompt_ia(contexto_mercado)

    try:
        response = client.chat.completions.create(
            model="deepseek/deepseek-v4-flash:free",
            messages=[
                {"role": "user", "content": prompt_final}
            ],
            max_tokens=300,
            temperature=0.2
        )

        if response and response.choices and response.choices[0].message:
            return response.choices[0].message.content
        else:
            # Log the full response object for debugging if choices are missing
            print(f"❌ API response did not contain expected choices or message: {response}")
            return json.dumps({"acao": "ESPERAR", "confianca": 0.0, "justificativa_curta": "API response missing choices or message."})
    except Exception as e:
        print(f"❌ Erro ao chamar a API da IA: {e}")
        return json.dumps({"acao": "ESPERAR", "confianca": 0.0, "justificativa_curta": f"Erro na chamada da API da IA: {e}"})


In [54]:
def analisar_mercado_wrapper(contexto_mercado):
    resposta_texto = analisar_mercado(contexto_mercado, client)
    return resposta_texto

In [55]:
decisao = analisar_mercado_wrapper(contexto_mercado)
print(decisao)

{
  "acao": "COMPRAR",
  "confianca": 0.65,
  "justificativa_curta": "Forte pressão compradora no livro de ofertas (bid/ask ratio 31.84) e alta pressão de compra ponderada (0.83), indicando suporte imediato. Tendência lateral de curto prazo, mas viés altista pela assimetria do order book."
}
